# FARSHORE | free cinematic audio generator

Use this notebook on Google Colab. It regenerates the bundled basic narration into higher-quality Kokoro neural speech and optionally replaces soundscapes with local Stable Audio 3 outputs. No paid API is used.

**Workflow:** upload the Farshore-Audio-Assets.zip when requested, run the Kokoro cells, then download the upgraded ZIP. Optional Stable Audio generation may take longer and requires enough free runtime. Choose a GPU runtime if available. No paid Colab tier is required by the code, but free compute quotas and model-download access are outside this notebook's control.

Source projects: [Kokoro](https://github.com/hexgrad/kokoro), [Stable Audio 3](https://github.com/Stability-AI/stable-audio-3).

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess, sys, os

work = Path('/content/farshore_audio_work')
work.mkdir(exist_ok=True)
uploaded = files.upload()
names = [n for n in uploaded if n.lower().endswith('.zip')]
assert len(names) == 1, 'Upload exactly Farshore-Audio-Assets.zip.'
source = Path(names[0])
with zipfile.ZipFile(source) as z:
    for member in z.infolist():
        target = (work / member.filename).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(work)
pack = work / 'Farshore-Audio-Assets'
assert (pack / 'audio-plan.json').is_file(), 'The expected starter package was not found.'
print('Ready:', pack)


## 1. Generate all 21 neural voiceovers with Kokoro

This section is the recommended minimum upgrade. The starter ZIP includes basic Flite speech only for immediate audio integration. Kokoro replaces those files in-place with neural dialogue using the same filenames and subtitles.

In [ ]:
subprocess.run(['apt-get', '-qq', '-y', 'install', 'espeak-ng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kokoro>=0.9.4', 'soundfile'], check=True)
subprocess.run([sys.executable, str(pack/'tools/render_kokoro.py')], check=True)


In [ ]:
from IPython.display import Audio, display
display(Audio(filename=str(pack/'audio/voice/intro.mp3')))
display(Audio(filename=str(pack/'audio/voice/oracle-ready.mp3')))


## 2. Optional: regenerate original scene music, ambience and or sound effects with Stable Audio 3

The procedural starter effects and music are already usable. This optional stage creates AI-generated audio in the local Colab runtime. Review [the Stability AI Community License](https://stability.ai/license). You may need to accept model terms for [Small-SFX](https://huggingface.co/stabilityai/stable-audio-3-small-sfx) and [Small-Music](https://huggingface.co/stabilityai/stable-audio-3-small-music) with a free Hugging Face account. If downloads are gated, add a read token in Colab's Secrets panel under HF_TOKEN.

Start with scope 'test' (one ambience and one music track). Change to 'scenes' for all 10 environments; 'all' also regenerates the 34 one-shot effects. Free Colab sessions are variable, and installing the optional model dependencies can take a while. Leave the option disabled if you just want the neural voices and starter sound effects.

In [ ]:
RUN_STABLE_AUDIO = False  # @param {type:"boolean"}
SCOPE = "test"  # @param ["test", "scenes", "all"]
SECONDS = 18  # @param {type:"integer"}

if RUN_STABLE_AUDIO:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception:
        pass
    repo = Path('/content/stable-audio-3')
    if not repo.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/Stability-AI/stable-audio-3.git',str(repo)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    subprocess.run(['uv','sync'],cwd=repo,check=True)
    subprocess.run(['uv','run','python',str(pack/'tools/render_stable.py'),'--scope',SCOPE,'--seconds',str(SECONDS)],cwd=repo,check=True)
else:
    print('Skipped optional AI music/SFX. Starter files are preserved.')


## 3. Download the completed audio pack

When the voiceovers are generated, run this final cell. The ZIP contains the exact audio filenames, the source plan, the updated generation-status metadata, and Claude integration instructions. Give this ZIP to Claude Code.

In [ ]:
import json
status = json.loads((pack/'generation-status.json').read_text())
from collections import Counter
for category, items in status.items():
    print(category, dict(Counter(items.values())))
archive = shutil.make_archive('/content/Farshore-Colab-Audio','zip',root_dir=pack.parent,base_dir=pack.name)
files.download(archive)
